# 02 · Análise exploratória — as oito fases

**CRISP-DM fases 2.3 e 2.4**

```
Base de dados
   ├─ 1ª  Distribuição dos atributos
   ├─ 2ª  Análise univariada
   ├─ 3ª  Análise bivariada
   ├─ 4ª  Análise multivariada
   ├─ 5ª  Valores ausentes
   ├─ 6ª  Valores discrepantes
   ├─ 7ª  Detecção de correlação
   └─ 8ª  Seleção de características
   ▼
Coleção de insights
```

Cada fase grava figura **e** a tabela que a originou. Como os notebooks são
versionados sem saída (`nbstripout`), o que não for persistido em disco desaparece do
repositório.

In [ ]:
import sys, warnings
from pathlib import Path

# O notebook roda a partir de `notebooks/`; a raiz do projeto entra no path
# para que `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from src import config
from src.utils.io import configurar_estilo

configurar_estilo()
print(f"Projeto em {RAIZ}")

In [ ]:
from src.data.ingestao import carregar_bruto
from src.data.preparacao import carregar_curada
from src.eda import fluxo8fases as eda

bruto = carregar_bruto()
curada = carregar_curada()
print(f"Bruto:  {bruto.shape[0]} × {bruto.shape[1]}")
print(f"Curada: {curada.shape[0]} × {curada.shape[1]}")

## 1ª fase — Distribuição dos atributos

In [ ]:
perfil = eda.fase_1_distribuicao_atributos(bruto, curada)
perfil.head(10)

## 2ª fase — Análise univariada

Distribuição, dispersão, assimetria e curtose de cada variável do espaço de atributos.
A coluna `assimetria` é a que decide quem recebe `log1p` na preparação.

In [ ]:
resumo = eda.fase_2_univariada(curada)
resumo[["mean", "50%", "std", "assimetria", "curtose", "pct_zeros"]]

## 3ª fase — Análise bivariada

Pearson não se aplica a variáveis sem ordem. Para as nominais, a medida é o **V de
Cramér**; para o par nominal × numérica, a **razão de correlação** (η²).

In [ ]:
cramer = eda.fase_3_bivariada(curada)
cramer.round(2)

## 4ª fase — Análise multivariada

O que aparece quando se olham todas as variáveis de uma vez. O PCA aqui é
**exploratório**: serve para ver estrutura, não para substituir o espaço original.

In [ ]:
variancia = eda.fase_4_multivariada(curada)
variancia.head(10)

## 5ª fase — Valores ausentes

A pergunta não é *quanto falta*, é **por que falta**. Colunas que faltam sempre juntas
denunciam ausência estrutural: são imóveis sem garagem e sem porão.

In [ ]:
ausentes = eda.fase_5_ausentes(bruto)
ausentes[["nulos", "pct_nulos", "natureza"]]

## 6ª fase — Valores discrepantes

Três métodos, porque cada um enxerga uma coisa: IQR e z robusto olham uma variável por
vez; o Isolation Forest olha a combinação. Um imóvel pode ter área normal e idade
normal e ainda assim ser atípico.

In [ ]:
discrepantes = eda.fase_6_discrepantes(curada)
discrepantes

## 7ª fase — Detecção de correlação

Variável redundante entra duas vezes na distância e pesa dobrado sem que ninguém tenha
decidido isso.

In [ ]:
redundantes = eda.fase_7_correlacao(curada)
redundantes

## 8ª fase — Seleção de características

Cada exclusão com motivo registrado. E uma redundância **mantida deliberadamente**:
área habitável e número de cômodos se correlacionam, mas medem coisas diferentes —
dois imóveis de mesma área com 5 e 9 cômodos são produtos diferentes.

In [ ]:
decisoes = eda.fase_8_selecao(curada, redundantes)
decisoes

## Coleção de insights

O produto da fase exploratória. Um insight sem **consequência** é curiosidade; a
terceira coluna é o que liga esta fase às decisões de preparação e modelagem.

In [ ]:
colecao = eda.executar()
print(f"{len(colecao)} insights registrados")
colecao[["fase", "insight"]]

In [ ]:
for _, linha in colecao.iterrows():
    print(f"[{linha.name}] {linha['fase']}")
    print(f"    achado:       {linha['insight']}")
    print(f"    consequência: {linha['consequencia']}")
    print(f"    evidência:    {linha['evidencia']}\n")